# Modelado — Cancelaciones de reservas hoteleras

- **Proyecto:** Práctica Final · Machine Learning y Deep Learning
- **Máster:** Inteligencia Artificial, Cloud Computing y DevOps · PontIA.tech
- **Fase:** 4 — Modelado

## Objetivo

Entrenar y evaluar 5 modelos de clasificación binaria para predecir cancelaciones:

1. LogisticRegression (baseline)
2. DecisionTreeClassifier
3. RandomForestClassifier
4. XGBClassifier (Gradient Boosting)
5. Red Neuronal con Keras

Todos los experimentos se registran en MLflow para comparación sistemática.

## Métrica principal

ROC-AUC (siguiendo la decisión tomada en Sección 2 del EDA).
Métrica secundaria: F1-score.

In [8]:
"""Setup del notebook de modelado."""

import sys
from pathlib import Path

# Path setup
PATH_PROYECTO = Path.cwd().parents[1] if "notebooks" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(PATH_PROYECTO))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Imports de nuestros módulos
from src.data_loader import preparar_datos, SEED
from src.models import (
    configurar_mlflow,
    calcular_metricas,
    imprimir_metricas,
)

print(f"Raíz del proyecto: {PATH_PROYECTO}")
print(f"SEED: {SEED}")

Raíz del proyecto: c:\Juan\Pontia\ML\practica-final-ml\practica-final-ml
SEED: 42


## Preparación de datos

Reusamos `preparar_datos()` del módulo `data_loader.py` (Fase 3). Una sola llamada
hace todo el preprocesamiento: carga, limpieza, encoding, escalado, split.

In [9]:
"""Cargar datos preprocesados."""

X_train, X_test, y_train, y_test, preprocesador = preparar_datos()

print(f"X_train: {X_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_train: {y_train.shape} (cancela: {y_train.mean()*100:.2f}%)")
print(f"y_test:  {y_test.shape} (cancela: {y_test.mean()*100:.2f}%)")

Eliminadas 13 filas con valores imposibles (0.011% del total).
X_train: (95501, 130)
X_test:  (23876, 130)
y_train: (95501,) (cancela: 37.04%)
y_test:  (23876,) (cancela: 37.04%)


## Setup de MLflow

MLflow es la herramienta de tracking de experimentos. Cada vez que entrenamos
un modelo, MLflow registra automáticamente:
- Hiperparámetros usados.
- Métricas obtenidas.
- El modelo entrenado (para poder recuperarlo después).

Los runs se guardan en la carpeta `mlruns/` del proyecto (ya añadida a `.gitignore`
en el siguiente paso para no commitearla).

In [10]:
"""Configurar MLflow."""

configurar_mlflow()

MLflow configurado.
  Tracking URI: file:///C:/Juan/Pontia/ML/practica-final-ml/practica-final-ml/mlruns
  Experimento: cancelaciones_hoteleras


## Modelo 1 — LogisticRegression (baseline)

Empezamos con el modelo más simple: regresión logística sin balanceo de clases.
Es nuestro baseline contra el que compararemos el resto de modelos.

**Por qué es el baseline**:

- Es un modelo lineal: aprende una combinación lineal de las features.
- Es rápido de entrenar (segundos en este dataset).
- Es interpretable: podemos ver qué peso aprende para cada variable.
- Cualquier modelo más complejo debe superarlo claramente para justificar
  su complejidad añadida.

**Limitación**:

- Solo capta relaciones LINEALES. Si la relación entre features y target
  es no lineal (interacciones complejas, umbrales, efectos no monótonos),
  no la detectará. Los modelos posteriores (RandomForest, XGBoost) sí.

**Hipótesis a priori**: ROC-AUC entre 0.82 y 0.86. Bueno pero mejorable.

In [12]:
"""Entrenar LogisticRegression (baseline)."""

import importlib
from src import models
importlib.reload(models)

from src.models import entrenar_logistic_regression, imprimir_metricas

modelo_lr, metricas_lr = entrenar_logistic_regression(
    X_train, y_train, X_test, y_test
)

imprimir_metricas(metricas_lr, "LogisticRegression (baseline)")

2026/05/27 13:35:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2026/05/27 13:35:21 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.



METRICAS: LogisticRegression (baseline)
  accuracy    : 0.8313
  precision   : 0.8110
  recall      : 0.7098
  f1          : 0.7570
  roc_auc     : 0.9102
